## Integrantes
- Andre Marroquin
- Gabriel Paz
- Mauricio lemus

# Fase B Modelo Federado e Inferencia sobre Banco 3

Este cuaderno implementa el Objetivo B del proyecto de detección de fraude federada. Se usan tres datasets de transacciones de tarjeta basadas en ISO 8583. Banco 1 y Banco 2 tienen la variable objetivo `is_fraud` y se usan para entrenamiento y validación. Banco 3 se trata como no etiquetado.

El objetivo es entrenar una simulación federada razonable usando Banco 1 y Banco 2 y generar inferencias binarias para Banco 3. Se generan inferencias preliminares para el primer 30 por ciento y una inferencia final para el 100 por ciento.

El enfoque es inferencial. Primero se revisan columnas tipos valores faltantes y estructura antes de modelar.


## Environment setup

Se preparan librerías rutas constantes y carpetas de salida. El cuaderno requiere Python 3.12 o superior.


In [1]:
from pathlib import Path
import json
import sys
import warnings
from datetime import datetime

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import average_precision_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import StandardScaler
try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings("ignore")
if sys.version_info < (3, 12):
    raise RuntimeError("Python 3.12 or higher is required")

random_seed = 42
np.random.seed(random_seed)

project_dir = Path.cwd()
data_dir = project_dir / "data"
output_dir = project_dir / "outputs_fase_B"
mini_eda_dir = output_dir / "01_mini_eda"
data_checks_dir = output_dir / "02_data_checks"
features_dir = output_dir / "03_features"
models_dir = output_dir / "04_models"
predictions_dir = output_dir / "05_predictions"
plots_dir = output_dir / "06_plots"
reports_dir = output_dir / "07_reports"

for folder in [output_dir, mini_eda_dir, data_checks_dir, features_dir, models_dir, predictions_dir, plots_dir, reports_dir]:
    folder.mkdir(parents=True, exist_ok=True)

bank_file_candidates = {
    "bank_1": ["Copia de 01_bo_vip_seed22_n100000.csv", "01_bo_vip_seed22_n100000.csv"],
    "bank_2": ["Copia de 02_br_privado_seed33_n100000.csv", "02_br_privado_seed33_n100000.csv"],
    "bank_3": ["Copia de 03_gt_estatal_seed3_n100000.csv", "03_gt_estatal_seed3_n100000.csv"],
}
target_column = "is_fraud"
preliminary_fraction = 0.30
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print(f"Project directory: {project_dir}")
print(f"Output directory: {output_dir}")
print(f"Python version: {sys.version.split()[0]}")


Project directory: d:\Proyecto-DataSience-PlusTI
Output directory: d:\Proyecto-DataSience-PlusTI\outputs_fase_B
Python version: 3.12.10


## Data loading helpers

Se definen funciones para localizar archivos en la raíz o en `data`, detectar separadores CSV, cargar datasets y normalizar nombres de columnas.

In [2]:
def normalize_to_ascii(value):
    return str(value).encode("ascii", errors="ignore").decode("ascii")


def to_snake_case(value):
    text = normalize_to_ascii(value).strip()
    text = pd.Series([text]).str.replace(r"[^0-9a-zA-Z]+", "_", regex=True).iloc[0]
    text = pd.Series([text]).str.replace(r"([a-z0-9])([A-Z])", r"\1_\2", regex=True).iloc[0]
    text = pd.Series([text]).str.replace(r"_+", "_", regex=True).iloc[0]
    return text.strip("_").lower() or "unnamed_column"


def normalize_column_names(columns):
    result = []
    seen = {}
    for column in columns:
        base = to_snake_case(column)
        count = seen.get(base, 0)
        result.append(base if count == 0 else f"{base}_{count + 1}")
        seen[base] = count + 1
    return result


def detect_csv_separator(path):
    sample = path.read_text(encoding="utf-8", errors="ignore")[:8192]
    counts = {sep: sample.count(sep) for sep in [";", ",", "\t", "|"]}
    best = max(counts, key=counts.get)
    return best if counts[best] > 0 else ","


def find_input_file(file_candidates):
    for folder in [project_dir, data_dir]:
        for file_name in file_candidates:
            path = folder / file_name
            if path.exists():
                return path
    raise FileNotFoundError("Input file not found")


def load_bank_dataset(bank_key, file_candidates):
    path = find_input_file(file_candidates)
    separator = detect_csv_separator(path)
    raw_data = pd.read_csv(path, sep=separator, low_memory=False)
    original_columns = list(raw_data.columns)
    normalized_columns = normalize_column_names(original_columns)
    data = raw_data.copy()
    data.columns = normalized_columns
    mapping = pd.DataFrame({"bank_key": bank_key, "original_column": original_columns, "normalized_column": normalized_columns})
    return {"bank_key": bank_key, "path": path, "separator": separator, "data": data, "mapping": mapping}


loaded_banks = {bank_key: load_bank_dataset(bank_key, candidates) for bank_key, candidates in bank_file_candidates.items()}
bank_1_data = loaded_banks["bank_1"]["data"]
bank_2_data = loaded_banks["bank_2"]["data"]
bank_3_data = loaded_banks["bank_3"]["data"]

load_summary = pd.DataFrame([
    {"bank_key": key, "file_name": item["path"].name, "separator": item["separator"], "row_count": len(item["data"]), "column_count": item["data"].shape[1]}
    for key, item in loaded_banks.items()
])
display(load_summary)


,bank_key,file_name,separator,row_count,column_count
0,bank_1,Copia de 01_bo_vip_seed22_n100000.csv,;,100003,66
1,bank_2,Copia de 02_br_privado_seed33_n100000.csv,;,100000,66
2,bank_3,Copia de 03_gt_estatal_seed3_n100000.csv,;,100000,66


## EDA inicial por archivo

Se ejecuta un miniEDA breve por banco para conocer forma columnas tipos nulos cardinalidad y estado inicial del target.


In [3]:
def has_valid_target_values(series):
    values = series.dropna().astype(str).str.strip().str.lower()
    values = values[values != ""]
    if values.empty:
        return False
    return set(values.unique()).issubset({"true", "false", "1", "0", "yes", "no", "y", "n"})


def target_distribution(series):
    table = series.value_counts(dropna=False).rename("count").reset_index()
    table.columns = ["target_value", "count"]
    table["rate"] = table["count"] / max(len(series), 1)
    return table


def create_mini_eda(bank_key, loaded_bank, show_target):
    data = loaded_bank["data"]
    has_target = target_column in data.columns
    valid_target = has_valid_target_values(data[target_column]) if has_target else False
    summary = pd.DataFrame({
        "bank_key": bank_key,
        "file_name": loaded_bank["path"].name,
        "separator": loaded_bank["separator"],
        "column_name": data.columns,
        "dtype": data.dtypes.astype(str).values,
        "missing_rate": data.isna().mean().values,
        "cardinality": data.nunique(dropna=False).values,
        "has_target": has_target,
        "target_has_valid_values": valid_target,
    })
    print("=" * 90)
    print(f"Dataset: {bank_key}")
    print(f"Rows: {len(data)}")
    print(f"Columns: {data.shape[1]}")
    print(f"Separator: {repr(loaded_bank['separator'])}")
    print(f"Has is_fraud: {has_target}")
    print(f"Valid is_fraud: {valid_target}")
    display(data.head())
    display(pd.Series(data.columns.tolist(), name="column_name").to_frame())
    display(data.dtypes.astype(str).rename("dtype").to_frame())
    display(data.isna().mean().rename("missing_rate").to_frame())
    display(data.nunique(dropna=False).rename("cardinality").to_frame())
    if show_target and has_target:
        display(target_distribution(data[target_column]))
    summary.to_csv(mini_eda_dir / f"{bank_key}_mini_eda_summary.csv", index=False)
    loaded_bank["mapping"].to_csv(mini_eda_dir / f"column_mapping_{bank_key}.csv", index=False)
    return summary


mini_eda_tables = [
    create_mini_eda("bank_1", loaded_banks["bank_1"], True),
    create_mini_eda("bank_2", loaded_banks["bank_2"], True),
    create_mini_eda("bank_3", loaded_banks["bank_3"], False),
]
load_summary["has_is_fraud"] = [target_column in loaded_banks[key]["data"].columns for key in load_summary["bank_key"]]
load_summary["target_has_valid_values"] = [has_valid_target_values(loaded_banks[key]["data"][target_column]) if target_column in loaded_banks[key]["data"].columns else False for key in load_summary["bank_key"]]
load_summary.to_csv(mini_eda_dir / "load_summary.csv", index=False)
display(load_summary)


Dataset: bank_1
Rows: 100003
Columns: 66
Separator: ';'
Has is_fraud: True
Valid is_fraud: True


,transaction_id,bank_code,bank_name,bank_country,bank_tier,client_id,client_segment,channel,card_brand,pan_masked,...,amount_usd,is_international,distance_from_home_km,hour_local,day_of_week,approved,response_description,client_baseline_amount,client_home_city,is_fraud
0,7dd812b1-bd03-4d05-afc6-c318dcc9b651,BO-VIP,BO-VIP,BO,vip,BO-VIP-CL-00001325,PLATINUM,POS,MASTERCARD,531270******3773,...,500.12,True,8717.0,20,Tue,True,Approved,2012.51,TARIJA,False
1,c08b49a6-889a-491a-a1f8-974526f7886d,BO-VIP,BO-VIP,BO,vip,BO-VIP-CL-00000079,PRIVATE,ECOM,VISA,421250******5552,...,1898.93,False,4.9,20,Tue,True,Approved,1096.46,LAPAZ,False
2,b04f88bd-2e33-42e5-a3cf-d52ef22dd7d9,BO-VIP,BO-VIP,BO,vip,BO-VIP-CL-00002344,INFINITE,ECOM,NaN,531270******6104,...,349.85,False,4.4,20,Tue,True,Approved,1528.37,SANTACRUZ,False
3,3a836c25-7a8c-473b-8141-3e84ba3f212d,BO-VIP,BO-VIP,BO,vip,BO-VIP-CL-00002587,PLATINUM,ATM,VISA,479500******0288,...,345.58,True,3966.0,20,Tue,True,Approved,2483.34,SUCRE,False
4,be9956da-924f-4c68-aed8-f0c5d949e577,BO-VIP,BO-VIP,BO,vip,BO-VIP-CL-00000087,PRIVATE,POS,VISA,479500******0249,...,118.90,False,348.0,20,Tue,True,NaN,1334.55,SUCRE,False


,column_name
0,transaction_id
1,bank_code
2,bank_name
3,bank_country
4,bank_tier
...,...
61,approved
62,response_description
63,client_baseline_amount
64,client_home_city


,dtype
transaction_id,object
bank_code,object
bank_name,object
bank_country,object
bank_tier,object
...,...
approved,bool
response_description,object
client_baseline_amount,float64
client_home_city,object


,missing_rate
transaction_id,0.00000
bank_code,0.00000
bank_name,0.00000
bank_country,0.00000
bank_tier,0.00000
...,...
approved,0.00000
response_description,0.00988
client_baseline_amount,0.00000
client_home_city,0.00000


...

,column_name
0,transaction_id
1,bank_code
2,bank_name
3,bank_country
4,bank_tier
...,...
61,approved
62,response_description
63,client_baseline_amount
64,client_home_city


,dtype
transaction_id,object
bank_code,object
bank_name,object
bank_country,object
bank_tier,object
...,...
approved,bool
response_description,object
client_baseline_amount,float64
client_home_city,object


,missing_rate
transaction_id,0.00000
bank_code,0.00000
bank_name,0.00000
bank_country,0.00000
bank_tier,0.00000
...,...
approved,0.00000
response_description,0.02995
client_baseline_amount,0.00000
client_home_city,0.00000


,cardinality
transaction_id,100000
bank_code,1
bank_name,1
bank_country,1
bank_tier,1
...,...
approved,2
response_description,13
client_baseline_amount,4503
client_home_city,8


,bank_key,file_name,separator,row_count,column_count,has_is_fraud,target_has_valid_values
0,bank_1,Copia de 01_bo_vip_seed22_n100000.csv,;,100003,66,True,True
1,bank_2,Copia de 02_br_privado_seed33_n100000.csv,;,100000,66,True,True
2,bank_3,Copia de 03_gt_estatal_seed3_n100000.csv,;,100000,66,True,False
